# RAG con LangChain

Este script implementa el mismo pipeline RAG del lab anterior
usando LangChain. El objetivo es ver cómo cada pieza del lab
mapea a una abstracción de LangChain.

**Mapeo lab → LangChain:**

| Lab (manual)           | LangChain                              |
|------------------------|----------------------------------------|
| `EmbeddingService`     | `GoogleGenerativeAIEmbeddings`         |
| `semantic_chunker()`   | `RecursiveCharacterTextSplitter`       |
| `ingest_documents()`   | `QdrantVectorStore.from_documents()`   |
| `retrieve()`           | `vectorstore.as_retriever()`           |
| `GenerationService`    | `ChatGoogleGenerativeAI`               |
| `ask()` manual         | LCEL pipeline con `|`                  |

**Stack:**
- `langchain-google-genai` — embeddings + LLM
- `langchain-qdrant`       — vector store
- `langchain`              — pipeline (LCEL)

In [1]:
!pip install -q langchain langchain-google-genai langchain-qdrant qdrant-client python-dotenv

In [2]:
import os
from typing import List, Dict, Tuple, Optional
from dotenv import load_dotenv
from pathlib import Path
import json

load_dotenv(override=True)
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
assert GOOGLE_API_KEY, "Falta GOOGLE_API_KEY en .env"
print("Setup completo")

Setup completo


### Datos — mismos documentos del lab

In [3]:
# ── Carga ────────────────────────────────────────────────────

def load_documents(path: str | Path) -> List[Dict]:
    """
    Carga documentos desde un archivo JSON.
    Valida que cada documento tenga los campos mínimos requeridos.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    with path.open(encoding="utf-8") as f:
        documents = json.load(f)

    required_fields = {"id", "title", "content"}
    for i, doc in enumerate(documents):
        missing = required_fields - doc.keys()
        if missing:
            raise ValueError(f"Documento {i} le faltan campos: {missing}")

    return documents

In [4]:
DOCS_PATH = Path("data/company_documents.json")  # ajusta la ruta si es necesario

company_documents = load_documents(DOCS_PATH)

for doc in company_documents:
    dept = doc.get("metadata", {}).get("department", "—")
    tipo = doc.get("metadata", {}).get("type", "—")
    print(f"  [{doc['id']}] {doc['title']}")
    print(f"           dept={dept}  type={tipo}  chars={len(doc['content'])}")

test_questions = [
    "Cual es el precio del plan Enterprise de CloudSync Pro?",
    "Cuantos dias a la semana pueden trabajar remotamente los empleados?",
    "Que pasa con los reembolsos mayores a $100?",
    "Que ocurre durante la primera semana de onboarding?",
]

print(f"{len(company_documents)} documentos cargados")

  [product_001] CloudSync Pro — Plan Enterprise
           dept=product  type=pricing  chars=372
  [policy_001] Política de Trabajo Remoto 2024
           dept=hr  type=policy  chars=350
  [process_001] Proceso de Reembolso a Clientes
           dept=support  type=process  chars=414
  [guide_001] Checklist de Onboarding para Nuevos Empleados
           dept=hr  type=guide  chars=344
4 documentos cargados


---
## Parte 1 — Embeddings

**En el lab:** `EmbeddingService(backend="gemini")` llamaba a
`client.models.embed_content()` directamente.

**En LangChain:** `GoogleGenerativeAIEmbeddings` es el wrapper equivalente.
La interfaz unificada es `embed_documents(texts)` y `embed_query(text)`.

In [5]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Lab: EmbeddingService(backend="gemini") con model="text-embedding-004"
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY,
)

# Prueba — equivalente a embedding_service.embed([...])
sample_vecs = embeddings.embed_documents([
    "El gato duerme en el sofa",
    "El felino descansa en el mueble",
])
print(f"Embeddings — dimension: {len(sample_vecs[0])}")

Embeddings — dimension: 3072


---
## Parte 2 — LLM de generacion

**En el lab:** `GenerationService(backend="gemini")` llamaba a
`client.models.generate_content()` directamente.

**En LangChain:** `ChatGoogleGenerativeAI` es el wrapper equivalente.
Sigue la interfaz `ChatModel` — recibe mensajes, retorna mensajes.
Todos los LLMs en LangChain tienen la misma interfaz: `llm.invoke(messages)`.

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

# Lab: GenerationService(backend="gemini") con model="gemini-2.5-flash"
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.1,
)

# Prueba
response = llm.invoke([HumanMessage(content="Di hola en una palabra.")])
print(f"LLM — respuesta: {response.content}")

LLM — respuesta: Hola


---
## Parte 3 — Chunking

**En el lab:** `semantic_chunker()` dividia por `\n\n` y acumulaba
hasta `max_chunk_size` caracteres.

**En LangChain:** `RecursiveCharacterTextSplitter` hace lo mismo —
intenta dividir por `\n\n`, luego `\n`, luego `.`, luego espacios.
Es el chunker mas usado en produccion con LangChain.

Ademas, LangChain trabaja con objetos `Document` en lugar de dicts.
`Document` tiene: `page_content` (texto) + `metadata` (dict).

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Lab: semantic_chunker(text, max_chunk_size=400)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=40,                        # solapamiento — el lab no tenia esto
    separators=["\n\n", "\n", ".", " "],     # orden de preferencia para dividir
)

# Convertir dicts del lab a Documents de LangChain
lc_documents: List[Document] = []
for doc in company_documents:
    chunks = text_splitter.split_text(doc["content"])
    for i, chunk in enumerate(chunks):
        lc_documents.append(Document(
            page_content=chunk,
            metadata={
                "source_id":  doc["id"],
                "title":      doc["title"],
                "chunk_idx":  i,
                **doc["metadata"],
            }
        ))

print(f"Chunking — {len(company_documents)} docs → {len(lc_documents)} chunks")
for d in lc_documents[:3]:
    print(f"  [{d.metadata['title'][:35]}] {d.page_content[:60]}...")

Chunking — 4 docs → 5 chunks
  [CloudSync Pro — Plan Enterprise] CloudSync Pro Enterprise ofrece almacenamiento ilimitado, co...
  [Política de Trabajo Remoto 2024] Vigente desde enero 2024. Los empleados pueden trabajar de f...
  [Proceso de Reembolso a Clientes] Paso 1: El cliente envía solicitud de reembolso por el porta...


---
## Parte 4 — Vector store con Qdrant

**En el lab:** `QdrantClient` + `PointStruct` + `upsert()` manual
con el loop de ingest_documents().

**En LangChain:** `QdrantVectorStore.from_documents()` hace
embed + upsert internamente en una sola llamada.

In [10]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

COLLECTION = "company_docs_lc_v2"

# Crear cliente y coleccion — igual que en el lab
#qdrant_client = QdrantClient(":memory:")
qdrant_client = QdrantClient(path = "./qdrant_data")
qdrant_client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=3072, distance=Distance.COSINE),
)

# Lab: ingest_documents() — embed + upsert en loop
# LangChain: from_documents() hace todo internamente
vectorstore = QdrantVectorStore(
    client=qdrant_client,
    collection_name=COLLECTION,
    embedding=embeddings,
)
vectorstore.add_documents(lc_documents)

print(f"Qdrant — {len(lc_documents)} chunks indexados en '{COLLECTION}'")

Qdrant — 5 chunks indexados en 'company_docs_lc_v2'


---
## Parte 5 — Retrieval

**En el lab:** `retrieve()` llamaba a `qdrant.query_points()` directamente
y retornaba una lista de dicts con `content`, `title`, `score`.

**En LangChain:** `vectorstore.as_retriever()` retorna un `Retriever`
— un objeto con `invoke(query)` que devuelve `List[Document]`.
El Retriever es la unidad basica del pipeline RAG en LangChain.

In [11]:
# Lab: retrieve(question, qdrant, emb_service, k=3, score_threshold=0.3)
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 3,
        "score_threshold": 0.3,
    },
)

# Prueba de retrieval
question = test_questions[0]
docs = retriever.invoke(question)
print(f"Pregunta: {question}")
print(f"Chunks recuperados: {len(docs)}")
for d in docs:
    print(f"  [{d.metadata.get('title', '')[:30]}] {d.page_content[:70]}...")

Pregunta: Cual es el precio del plan Enterprise de CloudSync Pro?
Chunks recuperados: 3
  [CloudSync Pro — Plan Enterpris] CloudSync Pro Enterprise ofrece almacenamiento ilimitado, colaboración...
  [Proceso de Reembolso a Cliente] . Reembolso completo disponible dentro de los 30 días posteriores a la...
  [Política de Trabajo Remoto 202] Vigente desde enero 2024. Los empleados pueden trabajar de forma remot...


---
## Parte 6 — Pipeline RAG completo con LCEL

**En el lab:** `ask()` hacia Retrieve → Augment → Generate manualmente
con un f-string como prompt.

**En LangChain:** LCEL (LangChain Expression Language) conecta los
componentes con el operador `|` (pipe). Es el patron moderno de LangChain.

```
Lab ask():                        LangChain LCEL:
chunks = retrieve(q)              retriever | format_docs
context = "\n---\n".join(chunks)       |
prompt = f"...{context}..."       ChatPromptTemplate
answer = generate(prompt)              |
return answer, chunks             llm | StrOutputParser()
```

In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


def format_docs(docs: List[Document]) -> str:
    """
    Convierte List[Document] a string de contexto.
    Equivalente al join en ask() del lab.
    """
    return "\n\n---\n\n".join(
        f"[{d.metadata.get('title', 'Sin titulo')}]\n{d.page_content}"
        for d in docs
    )


# Prompt template — equivalente al f-string en ask() del lab
prompt = ChatPromptTemplate.from_template("""
Eres un asistente empresarial. Responde la pregunta basandote UNICAMENTE
en el contexto proporcionado. Si la informacion no esta en el contexto,
di explicitamente que no la tienes.

CONTEXTO:
{context}

PREGUNTA: {question}

RESPUESTA:
""")

# ── LCEL pipeline ─────────────────────────────────────────
# El operador | encadena: cada componente recibe el output del anterior
# RunnablePassthrough() pasa el input sin modificar (la pregunta)
rag_chain = (
    {
        "context":  retriever | format_docs,  # retrieve → formatear a string
        "question": RunnablePassthrough(),     # pasar la pregunta tal cual
    }
    | prompt          # construir el prompt con contexto + pregunta
    | llm             # llamar al LLM
    | StrOutputParser()  # extraer el texto del mensaje de respuesta
)

# ── Prueba del pipeline completo ──────────────────────────
print("=" * 60)
print("RAG CON LANGCHAIN — PIPELINE COMPLETO")
print("=" * 60)

for question in test_questions:
    answer = rag_chain.invoke(question)
    print(f"\n Pregunta: {question}")
    print(f" Respuesta: {answer.strip()[:200]}")
    print("-" * 40)

RAG CON LANGCHAIN — PIPELINE COMPLETO

 Pregunta: Cual es el precio del plan Enterprise de CloudSync Pro?
 Respuesta: El precio del plan Enterprise de CloudSync Pro es de $49/mes por usuario con compromiso anual. El plan mensual sin compromiso está disponible a $59/mes por usuario.
----------------------------------------

 Pregunta: Cuantos dias a la semana pueden trabajar remotamente los empleados?
 Respuesta: Los empleados pueden trabajar de forma remota hasta 3 días por semana.
----------------------------------------

 Pregunta: Que pasa con los reembolsos mayores a $100?
 Respuesta: Los reembolsos mayores a $100 requieren aprobación del supervisor.
----------------------------------------

 Pregunta: Que ocurre durante la primera semana de onboarding?
 Respuesta: Durante la primera semana de onboarding, los nuevos empleados deben completar módulos de entrenamiento obligatorios sobre seguridad, cumplimiento y cultura.
----------------------------------------


---
## Parte 7 — Retornar fuentes junto a la respuesta

En el lab `ask()` retornaba `(answer, chunks)`.
En LangChain esto se hace con `RunnableParallel`.

In [13]:
from langchain_core.runnables import RunnableParallel

# Ejecutar retrieval y generacion en paralelo compartiendo el mismo input
rag_with_sources = RunnableParallel(
    answer=rag_chain,
    sources=(retriever | format_docs),
)

result = rag_with_sources.invoke(test_questions[0])
print(f"Respuesta:\n{result['answer'].strip()}")
print(f"\nContexto usado (primeros 300 chars):\n{result['sources'][:300]}...")

Respuesta:
El precio del plan Enterprise de CloudSync Pro es de $49/mes por usuario con compromiso anual. El plan mensual sin compromiso está disponible a $59/mes por usuario.

Contexto usado (primeros 300 chars):
[CloudSync Pro — Plan Enterprise]
CloudSync Pro Enterprise ofrece almacenamiento ilimitado, colaboración en tiempo real para hasta 500 usuarios y soporte prioritario 24/7. Precio: $49/mes por usuario con compromiso anual. Incluye: control de versiones avanzado, registros de auditoría, integración SS...


---
## Parte 8 — Tuning: cambiar parametros en runtime

En el lab se pasaban directamente a `retrieve()` y `ask()`.
En LangChain se crea un retriever con nuevos parametros.

In [33]:
# Threshold mas estricto — equivalente a ask(k=4, score_threshold=0.5)
retriever_strict = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 4, "score_threshold": 0.5},
)

rag_chain_strict = (
    {"context": retriever_strict | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

q = "Cual es la politica de trabajo remoto?"
print(f"THRESHOLD ESTRICTO (0.5):\n{rag_chain_strict.invoke(q).strip()}")

THRESHOLD ESTRICTO (0.5):
Los empleados pueden trabajar de forma remota hasta 3 días por semana. El trabajo remoto requiere aprobación del jefe directo. Hay un estipendio de equipos de $500 anuales para adecuación de oficina en casa. Las videollamadas son obligatorias en reuniones de equipo. El horario base de colaboración es de 10:00–15:00 hora local del empleado.
